# Session 6 - LightGBM vs MA4 vs blend, then freeze the Phase 1 forecast

Session 5 (D009) left a genuinely mixed result: LightGBM wins 8/15 rolling-CV origins and controls
bias ~3x better, MA4 wins the development-holdout point. This notebook settles which forecast to
freeze, using rolling-origin CV only (D008: the development holdout is not used for selection).

**Decision rule, written before any result was computed:**

- Candidates: LightGBM alone, MA4 alone, and an **equal-weight 50/50 blend**. Equal weights are
  fixed in advance rather than tuned, so no weight is fitted to the 15 validation origins (simple
  averages are routinely hard to beat with fitted weights - general forecast-combination practice,
  not VN1-specific hindsight, see D011).
- Choose the blend only if **(a)** its mean-of-origin score beats both components **and (b)** it
  wins at least 8/15 origins against each. Otherwise keep the better single model (LightGBM on a
  split, for its bias control).
- Other weights are reported as a diagnostic only and cannot change the choice.

In [1]:
import os
from pathlib import Path

current = Path.cwd()
while not (current / "CONTEXT.md").is_file():
    if current == current.parent:
        raise FileNotFoundError("Could not find CONTEXT.md in any parent directory")
    current = current.parent
os.chdir(current)

import numpy as np
import pandas as pd

from src.data import KEY
from src.features import build_direct_horizon_matrix, rolling_origin_folds
from src.metrics import vn1_score_by_group, vn1_score_breakdown
from src.models.lightgbm_model import train_lgbm
from src.pipeline import FROZEN_CONFIG, SEGMENT_DTYPE, frozen_forecast

df = pd.read_parquet("data/processed/vn1_long.parquet")

## 1. Rolling-origin CV - same 3 purged folds and FINAL_PARAMS as Session 5

In [2]:
origins = list(range(52, 144, 4))
horizons = list(range(1, 14))
folds = rolling_origin_folds(origins, horizons, n_folds=3)
weights = [0.0, 0.25, 0.5, 0.75, 1.0]  # share of LightGBM; 0.0 = MA4 alone, 1.0 = LightGBM alone

rows, fold_models = [], []
for fi, (fit_o, val_o) in enumerate(folds):
    fm = build_direct_horizon_matrix(df, fit_o, horizons)
    vm = build_direct_horizon_matrix(df, val_o, horizons)
    for m in (fm, vm):
        m["segment"] = m["segment"].astype(SEGMENT_DTYPE)
    feature_cols = [c for c in fm.columns if c not in (*KEY, "origin_date", "target_date", "target")]
    model, lgbm = train_lgbm(fm, vm, feature_cols, categorical_feature=["segment"],
                             origin_col="origin_date", **FROZEN_CONFIG["lgb_params"])
    fold_models.append((model, vm, feature_cols))
    ma4 = vm["roll_mean_4"].to_numpy()
    for w in weights:
        pred = w * lgbm + (1 - w) * ma4
        for origin, score in vn1_score_by_group(vm["target"], pred, vm["origin_date"]).items():
            mask = (vm["origin_date"] == origin).to_numpy()
            rows.append(dict(w_lgbm=w, origin=origin, score=score,
                             bias=vm.loc[mask, "target"].sum() - pred[mask].sum(),
                             actual=vm.loc[mask, "target"].sum()))
    print(f"fold {fi}: best_iteration={model.best_iteration_}")

cv = pd.DataFrame(rows)
by_origin = cv.pivot(index="origin", columns="w_lgbm", values="score")
by_origin.round(4)

fold 0: best_iteration=160


fold 1: best_iteration=283


fold 2: best_iteration=399


w_lgbm,0.00,0.25,0.50,0.75,1.00
origin,,,,,
2022-02-14,0.4283,0.4229,0.4264,0.4605,0.5076
2022-03-14,0.4190,0.4214,0.4318,0.4503,0.4745
2022-04-11,0.4650,0.4640,0.4706,0.4828,0.4998
2022-05-09,0.4500,0.4375,0.4478,0.4866,0.5310
2022-06-06,0.4642,0.4516,0.4486,0.4613,0.5071
2022-07-04,0.5109,0.4998,0.4927,0.4928,0.5020
2022-08-01,0.5256,0.5069,0.4956,0.4934,0.5019
2022-08-29,0.6556,0.6315,0.6142,0.6040,0.6009
2022-09-26,0.8400,0.7830,0.7407,0.7096,0.6876


In [3]:
summary = pd.DataFrame({
    "mean_of_origins": by_origin.mean(),
    "median": by_origin.median(),
    "mean_norm_abs_bias": cv.assign(nb=cv["bias"].abs() / cv["actual"]).groupby("w_lgbm")["nb"].mean(),
})
summary.round(4)

,mean_of_origins,median,mean_norm_abs_bias
w_lgbm,,,
0.00,0.5834,0.5025,0.0850
0.25,0.5562,0.4878,0.0651
0.50,0.5387,0.4812,0.0464
0.75,0.5366,0.4870,0.0349
1.00,0.5482,0.5076,0.0286


## 2. Apply the pre-registered rule

In [4]:
blend, lgbm_only, ma4_only = by_origin[0.5], by_origin[1.0], by_origin[0.0]
checks = {}
for name, other in [("LightGBM", lgbm_only), ("MA4", ma4_only)]:
    d = blend - other  # negative = blend better
    loo = [(d.sum() - x) / (len(d) - 1) for x in (d.min(), d.max())]
    checks[name] = dict(blend_wins=int((d < 0).sum()), n=len(d), mean_diff=d.mean(), median_diff=d.median(),
                        loo_excl_best_for_blend=loo[0], loo_excl_worst_for_blend=loo[1])
checks = pd.DataFrame(checks).T
print(checks.to_string(float_format="{:+.4f}".format))

rule_a = blend.mean() < min(lgbm_only.mean(), ma4_only.mean())
rule_b = (checks["blend_wins"] >= 8).all()
print(f"\n(a) blend mean beats both: {rule_a}   (b) >=8/15 wins vs each: {rule_b}")
print("ALL-SEASON RULE PICKS:", "50/50 blend" if rule_a and rule_b else "single model", "(see section 5 for Phase 1)")

          blend_wins        n  mean_diff  median_diff  loo_excl_best_for_blend  loo_excl_worst_for_blend
LightGBM    +11.0000 +15.0000    -0.0095      -0.0250                  -0.0043                   -0.0258
MA4         +11.0000 +15.0000    -0.0447      -0.0181                  -0.0262                   -0.0488

(a) blend mean beats both: True   (b) >=8/15 wins vs each: True
ALL-SEASON RULE PICKS: 50/50 blend (see section 5 for Phase 1)


**Rule satisfied: the 50/50 blend wins the all-season comparison.** It wins 11/15 origins against each component, and
its edge over LightGBM survives removing its single best origin (the leave-one-out mean diff stays
negative), unlike LightGBM's Session 5 edge over MA4, which mostly came from one origin. The
trade-off is bias: the blend's normalized |bias| sits between MA4's and LightGBM's, so part of
LightGBM's bias advantage is given back for lower per-series error. The 75/25 weight's slightly
lower mean is a diagnostic only, as pre-registered - its median is worse, and picking it would be
exactly the weight-fitting the rule excludes.

## 3. How optimistic is per-fold early stopping?

LightGBM early-stops on each fold's own validation origins, which flatters its validation score
(MA4 gets no such help). Bound it: re-score each fold at an iteration count taken from the *other*
folds' best iterations, which that fold's validation set had no say in. Capped at the trees the
model actually has (best_iteration + 50), so a fold whose cap binds is not a clean check.

In [5]:
bests = [m.best_iteration_ for m, _, _ in fold_models]
for fi, (m, vm, feature_cols) in enumerate(fold_models):
    want = int(np.median([b for j, b in enumerate(bests) if j != fi]))
    n = min(want, m.booster_.num_trees())
    lgbm = np.clip(m.predict(vm[feature_cols], num_iteration=n), 0, None)
    ma4 = vm["roll_mean_4"].to_numpy()
    fold_origins = sorted(vm["origin_date"].unique())
    for w in (1.0, 0.5):
        other = vn1_score_by_group(vm["target"], w * lgbm + (1 - w) * ma4, vm["origin_date"]).mean()
        print(f"fold {fi} w_lgbm={w}: early-stopped {by_origin.loc[fold_origins, w].mean():.4f}  "
              f"other-folds iter ({n}{', CAPPED' if n < want else ''}) {other:.4f}")

fold 0 w_lgbm=1.0: early-stopped 0.5040  other-folds iter (160, CAPPED) 0.5040
fold 0 w_lgbm=0.5: early-stopped 0.4450  other-folds iter (160, CAPPED) 0.4450


fold 1 w_lgbm=1.0: early-stopped 0.5982  other-folds iter (279) 0.5984
fold 1 w_lgbm=0.5: early-stopped 0.6172  other-folds iter (279) 0.6173


fold 2 w_lgbm=1.0: early-stopped 0.5424  other-folds iter (221) 0.5463
fold 2 w_lgbm=0.5: early-stopped 0.5539  other-folds iter (221) 0.5564


Where the check is valid, the optimism is at most a few thousandths - far smaller than the
blend's margin over either component.

## 4. One final evaluation of the frozen pipeline on the development holdout

`src/pipeline.py::frozen_forecast` is the exact code that generates the Phase 1 forecast; only the
forecast origin differs (156 here, 169 for Phase 1). Run once, after the decision above was
frozen - it cannot change it (D008). Still a development holdout, inspected since Session 3.

In [6]:
fc, model, info = frozen_forecast(df, 156)
print("forecast origin:", info["forecast_origin_date"], "| val origins:", info["val_origins"],
      "| best_iteration:", info["best_iteration"], "| tracked:", info["tracked_metric"])
pd.DataFrame({c: vn1_score_breakdown(fc["target"], fc[c]) for c in ("lgbm", "ma4", "blend")}).T.round(4)

forecast origin: 2023-07-03 | val origins: [132, 136, 140] | best_iteration: 491 | tracked: ['vn1_score_origin_mean']


,total_actual,total_predicted,signed_bias,abs_error_sum,bias_term,score
lgbm,3508827.0,3.467481e+06,41346.0416,1.711494e+06,41346.0416,0.4996
ma4,3508827.0,3.453011e+06,55815.7500,1.694063e+06,55815.7500,0.4987
blend,3508827.0,3.460246e+06,48580.8958,1.649414e+06,48580.8958,0.4839


LightGBM (0.4996) and MA4 (0.4987) reproduce Session 5 exactly, which confirms the refactored
pipeline matches the code that was evaluated before. The blend scores **0.4839**, ahead of both -
consistent with the all-season rolling CV. (The development holdout covers Jul-Oct, not Oct-Jan.)

## 5. Season-matched backtest - which model is best for an Oct-Jan forecast?

Checking the Phase 1 forecast before committing it showed LightGBM forecasting ~24% more demand
than MA4 over Oct-Jan. Phase 0 history supports a rise of that size: Oct-Jan demand ran 26-37%
above the preceding 13 weeks in every year (2020 +37%, 2021 +26%, 2022 +33%). MA4 is flat by
construction and cannot anticipate it, and the rolling-CV origins covering Q4 2022 already
favoured LightGBM alone (4 of 5) - but those folds had less training history than Phase 1 will.

So the question "which model is best *for Phase 1*" is answered directly: run the exact frozen
pipeline at the same calendar point one year earlier - origins 2022-09-19 to 2022-10-17 (Phase 1's
origin is 2023-10-02). Same code, same training procedure, same Oct-Jan window, all inside Phase 0
and before the development holdout. Decided in advance: lowest mean score wins, origin-level wins as
a check. Not D011 hindsight - this uses only Phase 0 data any 2024 competitor had.

In [7]:
dates = np.sort(df["date"].unique())
season_rows = []
for o in range(115, 120):
    fc_o, _, info_o = frozen_forecast(df, o)
    for c in ("lgbm", "blend", "ma4"):
        b = vn1_score_breakdown(fc_o["target"], fc_o[c])
        season_rows.append(dict(origin=pd.Timestamp(dates[o]).date(), model=c, score=b["score"],
                                norm_bias=b["signed_bias"] / b["total_actual"]))
season = pd.DataFrame(season_rows)
season_scores = season.pivot(index="origin", columns="model", values="score")
print(season.pivot(index="origin", columns="model", values=["score", "norm_bias"]).round(4).to_string())
print("mean:", season_scores.mean().round(4).to_dict())
print("origins won:", season_scores.idxmin(axis=1).value_counts().to_dict())

             score                 norm_bias                
model        blend    lgbm     ma4     blend    lgbm     ma4
origin                                                      
2022-09-19  0.7632  0.7308  0.8376    0.1979  0.1015  0.2942
2022-09-26  0.7498  0.7052  0.8400    0.1664  0.0451  0.2876
2022-10-03  0.7613  0.7170  0.8455    0.1645  0.0539  0.2750
2022-10-10  0.7505  0.7052  0.8395    0.1509  0.0437  0.2580
2022-10-17  0.7435  0.7055  0.8293    0.1425  0.0473  0.2377
mean: {'blend': 0.7536, 'lgbm': 0.7127, 'ma4': 0.8384}
origins won: {'lgbm': 5}


**LightGBM alone wins all 5 season-matched origins** (mean 0.7127 vs. blend 0.7536 vs. MA4
0.8384). The blend under-forecasts by 14-20% of demand here (positive norm_bias = under), LightGBM by
4-10%. Caveat: the 5 origins span one season and reuse only two distinct trained models (the
step-4 origin grid), so this is close to one strong data point, not five independent ones - but it
agrees with three years of Q4 rises and with the mechanism.

**Decision (D012):** the two results answer different questions. Averaged over all seasons, the
blend is best; for an Oct-Jan horizon, LightGBM alone is. Phase 1 is Oct-Jan, so **the primary
frozen Phase 1 forecast is LightGBM alone**; the blend and MA4 are frozen alongside as declared
secondaries and all three are scored when actuals arrive. The pre-registered all-season rule
(section 2) is superseded for Phase 1 by this season-matched comparison, and D012 records that.

## 6. Phase 1 forecast

Generated outside this notebook by `scripts/make_phase1_forecast.py`, which refuses to run with
uncommitted code changes and writes `submissions/phase1/phase1_forecast.csv` (primary) plus the
two secondaries and `phase1_provenance.json` (commit hash, config, best iteration, feature list,
input/output hashes, timestamp). Real Phase 1 actuals are not opened until those are committed.